In [1]:
import pandas as pd
import numpy as np
from collections import Counter
df = pd.read_csv("/Users/serhatguldogan/Library/CloudStorage/OneDrive-KocUniversitesi/Project_/RAW DATA/BTC_1sec.csv")


We do the same things that we did at "4_states_transitions_and_means.ipynb". Refer to that.

In [4]:
# Calculate price changes
df['price_change'] = df['midpoint'].diff()

# Function to discretize price changes for 16 states (buckets of 2.5)
def discretize_price_change_16_states(p):
    """
    Discretize price changes into 16 states (8 positive, 8 negative):
    - 0 < p ≤ 2.5 → 2.5
    - 2.5 < p ≤ 5 → 5
    - 5 < p ≤ 7.5 → 7.5
    - 7.5 < p ≤ 10 → 10
    - 10 < p ≤ 12.5 → 12.5
    - 12.5 < p ≤ 15 → 15
    - 15 < p ≤ 17.5 → 17.5
    - 17.5 < p ≤ 20 → 20
    Same for negatives (symmetric)
    - 0 stays 0
    """
    bucket_size = 2.5
    max_state = 8  # 8 positive states
    
    if pd.isna(p) or p == 0:
        return 0
    elif p > 0:
        if p > 100:
            return 100  # Cap at 20
        # Find which bucket (1-indexed)
        bucket = int(np.ceil(p / bucket_size))
        if bucket > max_state:
            bucket = max_state
        return bucket * bucket_size
    else:  # p < 0
        if p < -100:
            return -100  # Cap at -20
        # Find which bucket (1-indexed)
        bucket = int(np.ceil(abs(p) / bucket_size))
        if bucket > max_state:
            bucket = max_state
        return -bucket * bucket_size

# Apply discretization
df['price_change_discretized'] = df['price_change'].apply(discretize_price_change_16_states)

# Create label column: map discretized values to labels
# +2.5 → +1, +5 → +2, ..., +20 → +8
# -2.5 → -1, -5 → -2, ..., -20 → -8
# 0 → 0
def label_price_change_16_states(discretized_value):
    if discretized_value == 0:
        return 0
    elif discretized_value > 0:
        return int(discretized_value / 2.5)
    else:  # discretized_value < 0
        return int(discretized_value / 2.5)

df['price_change_label'] = df['price_change_discretized'].apply(label_price_change_16_states)

# Display results
print("First 20 rows with price changes:")
print(df[['midpoint', 'price_change', 'price_change_discretized', 'price_change_label']].head(20))
print("\nPrice change label distribution:")
print(df['price_change_label'].value_counts().sort_index())


First 20 rows with price changes:
     midpoint  price_change  price_change_discretized  price_change_label
0   56035.995           NaN                       0.0                   0
1   56035.995         0.000                       0.0                   0
2   56035.995         0.000                       0.0                   0
3   56035.995         0.000                       0.0                   0
4   56035.995         0.000                       0.0                   0
5   56035.995         0.000                       0.0                   0
6   56035.995         0.000                       0.0                   0
7   56035.995         0.000                       0.0                   0
8   56035.995         0.000                       0.0                   0
9   56035.995         0.000                       0.0                   0
10  56035.995         0.000                       0.0                   0
11  56035.995         0.000                       0.0                   0
12  

In [5]:
changes = df["price_change_label"]


In [6]:
# Initialize transition counter for 16 states: -8 to -1, 1 to 8
transition_counter = Counter()
for i in range(-8, 0):
    for j in range(-8, 9):
        if j != 0:
            transition_counter[(i, j)] = 0
for i in range(1, 9):
    for j in range(-8, 9):
        if j != 0:
            transition_counter[(i, j)] = 0

# Count transitions
for i in range(1, len(changes)):
    prev_state = changes.iloc[i - 1]
    curr_state = changes.iloc[i]
    # Only count if neither state is 0
    if prev_state != 0 and curr_state != 0:
        transition_counter[(prev_state, curr_state)] += 1

counter = dict(transition_counter)


In [7]:
# Remove transitions involving 0 (shouldn't be any, but just to be safe)
d = counter.copy()
for key in list(counter.keys()):
    if 0 in key:
        d.pop(key)

print(f"Total transitions: {sum(d.values())}")
print(f"Number of transition types: {len(d)}")


Total transitions: 298313
Number of transition types: 302


In [8]:
# Normalize transition probabilities by first state
normalized_by_first = {}
for a in set(key[0] for key in d.keys()):
    sum_ax = sum(v for (x, y), v in d.items() if x == a)
    if sum_ax == 0:
        continue
    for (x, y), v in d.items():
        if x == a:
            normalized_by_first[(x, y)] = v / sum_ax

print("Sample normalized transitions (first 10):")
for i, (k, v) in enumerate(list(normalized_by_first.items())[:10]):
    print(f"{k}: {v:.6f}")


Sample normalized transitions (first 10):
(1, -8): 0.008286
(1, -7): 0.003104
(1, -6): 0.004243
(1, -5): 0.007304
(1, -4): 0.012415
(1, -3): 0.027493
(1, -2): 0.080500
(1, -1): 0.292884
(1, 1): 0.324862
(1, 2): 0.110186


In [9]:
# Verify normalization (should sum to 1 for each starting state)
print("\nVerification - sums by starting state:")
for state in sorted(set(key[0] for key in normalized_by_first.keys())):
    total = sum(v for (x, y), v in normalized_by_first.items() if x == state)
    print(f"State {state:3d}: {total:.10f}")



Verification - sums by starting state:
State -40: 1.0000000000
State  -8: 1.0000000000
State  -7: 1.0000000000
State  -6: 1.0000000000
State  -5: 1.0000000000
State  -4: 1.0000000000
State  -3: 1.0000000000
State  -2: 1.0000000000
State  -1: 1.0000000000
State   1: 1.0000000000
State   2: 1.0000000000
State   3: 1.0000000000
State   4: 1.0000000000
State   5: 1.0000000000
State   6: 1.0000000000
State   7: 1.0000000000
State   8: 1.0000000000
State  40: 1.0000000000


In [10]:
# Create transition matrix (16x16)
# Order: [-8, -7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8]
indices = list(range(-8, 0)) + list(range(1, 9))
size = len(indices)
mat = np.zeros((size, size))

# Fill matrix
for i, row in enumerate(indices):
    for j, col in enumerate(indices):
        key = (row, col)
        if key in normalized_by_first:
            mat[i, j] = normalized_by_first[key]

print("Transition matrix shape:", mat.shape)
print("\nFirst few rows and columns:")
print(mat[:4, :4])


Transition matrix shape: (16, 16)

First few rows and columns:
[[0.13403832 0.03045915 0.03561099 0.04934924]
 [0.08398004 0.02716186 0.03519956 0.04711752]
 [0.08212255 0.02505791 0.03537587 0.05032638]
 [0.07736563 0.02134746 0.03588191 0.04663134]]


In [18]:
np.savetxt("mat_16_states.csv",X=mat,fmt="%f")

In [19]:
# Solve for stationary distribution: vP = v
# v(P - I) = 0 with constraint sum(v) = 1
P = mat
A = P.T - np.eye(size)

# Append normalization constraint
A = np.vstack([A, np.ones(size)])
b = np.zeros(size + 1)
b[-1] = 1

# Solve using least squares
v, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
v = v.real

v = np.array(v)

print("Stationary distribution:")
for i, state in enumerate(indices):
    print(f"State {state:3d}: {v[i]:.6f}")

print(f"\nSum of probabilities: {v.sum():.10f}")


Stationary distribution:
State  -8: 0.027231
State  -7: 0.009277
State  -6: 0.012495
State  -5: 0.017704
State  -4: 0.026700
State  -3: 0.046889
State  -2: 0.103364
State  -1: 0.230965
State   1: 0.263731
State   2: 0.112912
State   3: 0.052430
State   4: 0.029943
State   5: 0.018886
State   6: 0.012599
State   7: 0.009154
State   8: 0.025720

Sum of probabilities: 0.9999999966


In [21]:
np.savetxt("pi_16_states.csv",X = v, fmt="%f")

In [13]:
# Calculate mean price change from stationary distribution
# Each state j corresponds to price change = j * 2.5
bucket_size = 2.5

# Calculate mean: E[X] = Σ (state_j * price_change_j * probability_j)
# where price_change_j = state_j * bucket_size
mean_price_change = 0.0

print("State | Price Change | Probability | Contribution")
print("-" * 60)
for i, state in enumerate(indices):
    price_change = state * bucket_size
    prob = v[i]
    contribution = price_change * prob
    mean_price_change += contribution
    if abs(contribution) > 0.001:  # Only print significant contributions
        print(f"{state:5d} | {price_change:11.2f} | {prob:10.6f} | {contribution:12.6f}")

print("-" * 60)
print(f"\nMean price change (from stationary distribution): {mean_price_change:.6f}")


State | Price Change | Probability | Contribution
------------------------------------------------------------
   -8 |      -20.00 |   0.027231 |    -0.544614
   -7 |      -17.50 |   0.009277 |    -0.162356
   -6 |      -15.00 |   0.012495 |    -0.187432
   -5 |      -12.50 |   0.017704 |    -0.221294
   -4 |      -10.00 |   0.026700 |    -0.266998
   -3 |       -7.50 |   0.046889 |    -0.351664
   -2 |       -5.00 |   0.103364 |    -0.516820
   -1 |       -2.50 |   0.230965 |    -0.577413
    1 |        2.50 |   0.263731 |     0.659327
    2 |        5.00 |   0.112912 |     0.564562
    3 |        7.50 |   0.052430 |     0.393227
    4 |       10.00 |   0.029943 |     0.299431
    5 |       12.50 |   0.018886 |     0.236077
    6 |       15.00 |   0.012599 |     0.188983
    7 |       17.50 |   0.009154 |     0.160191
    8 |       20.00 |   0.025720 |     0.514393
------------------------------------------------------------

Mean price change (from stationary distribution): 0.187600
